# Casual-to-Spatial Perseiver Bridge

In [ ]:
# @title Setup
import os
import sys
from google.colab import drive

drive.mount("/content/drive")

WORKSPACE_DIR = "/content/drive/MyDrive/CSPB_Trainer"
os.makedirs(WORKSPACE_DIR, exist_ok=True)
%cd {WORKSPACE_DIR}

REPO_URL = "https://github.com/cosmicoxytocin/qwen-sdxl-adapter.git"  # @param {type:"string"}
REPO_DIR = os.path.join(WORKSPACE_DIR, "qwen-sdxl-adapter")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] += f":{os.path.expanduser('~/.cargo/bin')}"
!uv pip install --system  -r requirements.txt

In [ ]:
# @title WanDB Login
# @markdown Please **store your WanDB API key in Colab's secret storage** with key `WANDB_API` and value as your API key.
import os
import wandb
from google.colab import userdata

WANDB_API_KEY = userdata.get("WANDB_API")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    wandb.login(key=WANDB_API_KEY)
else:
    print("WANDB_API key not found in Colab secrets. Please set it up to enable WanDB logging.")

In [ ]:
# @title Huggingface Login
# @markdown Please **store your Huggingface API token in Colab's secret storage** as `HF_TOKEN` with value as your Huggingface API token.
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print(
        "Huggingface API token not found in Colab secrets. Please set it up to enable Huggingface logging."
    )

In [ ]:
#@title Data Configuration
import os

DATA_DIR = os.path.join(WORKSPACE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
CACHE_DIR = os.path.join(DATA_DIR, "cache")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"[INFO] Please ensure your images and text files are placed in:\n{RAW_DIR}\n")

In [ ]:
#@title Verify Data Files
import os
import glob


os.chdir(f"{WORKSPACE_DIR}/data/raw")

extensions = ("*.jpg", "*.jpeg", "*.png", "*.webp")
image_files = []
for ext in extensions:
    image_files.extend(glob.glob(os.path.join(RAW_DIR, ext)))

if not image_files:
    print("[WARNING] No images found in the raw data directory. Please upload your images to proceed.")
else:
    print(f"[INFO] Found {len(image_files)} image(s).")
    missing_txts = 0
    for img_path in image_files:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        txt_path = os.path.join(RAW_DIR, f"{base_name}.txt")
        if os.path.exists(txt_path):
            with open(txt_path, "r", encoding="utf-8") as f:
                prompt = f.read().strip()
            print(f"  - {base_name}: paired with prompt ({len(prompt)} chars)")
        else:
            print(f"  - {base_name}: [WARNING] No corresponding .txt file found.")
            missing_txts += 1
    if missing_txts == 0:
        print("\n[INFO] All images have corresponding text files.")


In [ ]:
#@title AoT Caching
import os

os.chdir(REPO_DIR)

!python scripts/cache_dataset.py \
    --raw_data_dir {RAW_DIR} \
    --cache_dir {CACHE_DIR}


In [ ]:
#@title Training [SmokeTest]
!python scripts/train.py \
    training.gradient_accumulation_steps=2 \
    training.max_train_steps=800 \
    training.learning_rate=1e-4 \
    training.mixed_precision="bf16" \
    data.batch_size=2 \
    data.cache_dir={CACHE_DIR} \
    data.caption_dropout_prob=0.1 \
    logging.run_name="cspb-smoketest"


In [ ]:
#@title Inference
import os
import IPython.display as display
os.chdir(REPO_DIR)

ckpt_dir = "./checkpoints"
checkpoints = [os.path.join(ckpt_dir, f)
               for f in os.listdir(ckpt_dir) if f.endswith(".safetensors")]
latest_ckpt = max(checkpoints, key=os.path.getmtime)
print(f"[INFO] Using checkpoint: {latest_ckpt}")

output_image_path = "./outputs/inference_result.png"
# TODO: Save output image with index to avoid overwriting

prompt = "" #@param {type:"string"}

!python scripts/inference.py \
    --prompt f"{prompt}" \
    --adapter_ckpt {latest_ckpt} \
    --steps 20 \
    --cfg 4.5 \
    --output {output_image_path}

display.Image(output_image_path)